# VARIANT-1 Experiment on Google Colab A100

This notebook runs the complete VARIANT-1 experiment with all improvements:
- Exponential deadline pressure
- Zone capacity constraints  
- Heavy task emphasis
- Burst arrivals
- Large-scale training (200k steps, 16 envs)

**Runtime**: ~4-6 hours on A100 GPU

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Navigate to project (adjust path as needed)
import os
os.chdir('/content/drive/MyDrive/LogitHMARL')
!pwd
!ls -la

In [ ]:
# 3. Check GPU availability
!nvidia-smi

In [ ]:
# 4. Install dependencies
!pip install -q torch torchvision torchaudio
!pip install -q stable-baselines3
!pip install -q gymnasium
!pip install -q pandas matplotlib seaborn
!pip install -q imageio

In [ ]:
# 5. Verify installation
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

In [ ]:
# 6. Set MODE to full for complete training
import os
os.environ['MODE'] = 'full'
print("MODE set to: full")
print("Configuration:")
print("  - Training steps: 200,000")
print("  - Batch size: 2,048")
print("  - Parallel envs: 16")
print("  - Eval steps: 300")

In [ ]:
# 7. Run the experiment (this will take 4-6 hours)
# Press Ctrl+C to interrupt if needed
!python run_experiments.py

## Alternative: Run in Background with nohup

In [ ]:
# Run in background (allows you to do other things)
%%bash
export MODE=full
nohup python run_experiments.py > variant1_colab.log 2>&1 &
echo "Experiment started in background. PID: $!"
echo "Check progress with: tail -f variant1_colab.log"

In [ ]:
# Monitor progress
!tail -100 variant1_colab.log

In [ ]:
# Check if still running
!ps aux | grep "python run_experiments.py" | grep -v grep

In [ ]:
# Monitor GPU usage
!nvidia-smi

## View Results

In [ ]:
# Load and display results
import pandas as pd

results = pd.read_csv('results/results.csv')
results_sorted = results.sort_values('total_value', ascending=False)

print("\n" + "="*80)
print("VARIANT-1 EXPERIMENT RESULTS")
print("="*80)
print(results_sorted.to_string(index=False))
print("\n")

# Highlight NL-HMARL vs S-Shape
nl_score = results[results['method'] == 'NL-HMARL']['total_value'].values[0]
ss_score = results[results['method'] == 'S-Shape']['total_value'].values[0]
print(f"NL-HMARL: {nl_score:,}")
print(f"S-Shape:  {ss_score:,}")
print(f"Difference: {nl_score - ss_score:,} ({((nl_score/ss_score - 1)*100):.1f}%)")

In [ ]:
# Visualize results
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.barh(results_sorted['method'], results_sorted['total_value'])
plt.xlabel('Total Value')
plt.title('VARIANT-1 Performance Comparison')
plt.tight_layout()
plt.savefig('results/performance_comparison.png', dpi=150)
plt.show()

## Download Results

In [ ]:
# Download results CSV
from google.colab import files
files.download('results/results.csv')

In [ ]:
# Package all results for download
!zip -r variant1_results.zip results/
files.download('variant1_results.zip')

## Troubleshooting

In [ ]:
# If you need to restart, first check for running processes
!ps aux | grep python

In [ ]:
# Kill a stuck process (replace PID with actual process ID)
# !kill -9 PID

In [ ]:
# Check disk space
!df -h

In [ ]:
# View Python error traceback if crash occurred
!tail -200 variant1_colab.log